# P229: Kaufsignal für Heizöl und Diesel – Gesamtprüfung

**Ziel:** Ein tägliches Signal für den Einkauf: **Kaufen**, **Warten** oder **Beobachten**. Das Modell (XGBoost) schätzt, ob der Preis in **3 Tagen höher** ist als heute.

**Dieses Notebook zeigt alles in einem Durchlauf. Einfach „Run all“:**

| Teil | Inhalt |
|---|---|
| **A** | Welche Daten und welche Spalten nutzt das Modell? |
| **B** | Wie wurde getestet (ohne Blick in die Zukunft)? |
| **C** | Fehlermaße: Trefferquote, Precision, Recall, Konfusionsmatrix, Kalibrierung, Kosten |
| **D** | Dritte Stufe „Beobachten“ |
| **E** | **Neue Tage, die das Modell noch nie gesehen hat** |
| **F** | Was das Ergebnis nicht zeigt |

**Dauer:** ca. 10 Minuten, wenn die gesicherten Vorhersagen aus db_10 im Output-Ordner liegen. Sonst werden sie neu berechnet (zusätzlich bis zu 30 Minuten). Teil E braucht ca. 5–10 Minuten.

**Wichtig vorab:** Ergebnisse aus Teil C und D gelten für den Zeitraum bis 20.01.2026, der schon oft ausgewertet wurde. Teil E prüft auf Tagen danach.

In [ ]:
%pip install xgboost scikit-learn openpyxl

In [ ]:
dbutils.library.restartPython()

In [ ]:
# Zelle 0: Legt den Projektcode (Ordner p229) auf dem Cluster an. Nichts ändern, nur ausführen.
import os
os.makedirs("/tmp/p229_code/p229", exist_ok=True)
FILES = {}
FILES['__init__'] = r'''"""P229: Kauf/Warten-Signal für Heizöl und Diesel (BayWa). Logik aus db_08_xgboost_sauber."""
'''
FILES['config'] = r'''"""Konstanten aus db_08 (unverändert übernommen)."""
import pandas as pd

SERIES = ["heizoel_sued", "heizoel_suedwest", "heizoel_suedost", "heizoel_rheinmain",
          "diesel_sued", "diesel_suedwest", "diesel_suedost", "diesel_rheinmain"]
EXO = ["ice_gasoil", "brent", "wti", "nymex_heating_oil", "usd_eur"]
LAST_ARGUS = pd.Timestamp("2026-01-20")   # danach OMR + fortgeschriebener ICE-Preis
DEV = (pd.Timestamp("2023-04-21"), pd.Timestamp("2024-11-01"))
TEST = (pd.Timestamp("2024-11-15"), pd.Timestamp("2026-01-20"))
HORIZON = 3
REG = dict(max_depth=2, n_estimators=150, learning_rate=0.03, min_child_weight=10,
           reg_lambda=10, subsample=0.7, colsample_bytree=0.5,
           eval_metric="logloss", random_state=42)
# Szenario -> (entry_lag, feat_lag)
SCEN = {"S0 (Kauf zu Preis t)": (0, 0),
        "S1 (Kauf zu Preis t+1)": (1, 0),
        "S2 (Daten nur bis t-1)": (0, 1)}

SERIEN_NAMEN = {"heizoel_sued": "Heizöl Süd", "heizoel_suedwest": "Heizöl Südwest", "heizoel_suedost": "Heizöl Südost",
                "heizoel_rheinmain": "Heizöl Rhein-Main", "diesel_sued": "Diesel Süd", "diesel_suedwest": "Diesel Südwest",
                "diesel_suedost": "Diesel Südost", "diesel_rheinmain": "Diesel Rhein-Main"}
'''
FILES['stats'] = r'''from math import sqrt

import numpy as np
from scipy.stats import binomtest


def wil(k, n, z=1.96):
    """Wilson-95-%-Intervall für eine Trefferquote k/n."""
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return c - h, c + h


def block_boot_mean(x, block=5, n_boot=2000, seed=42):
    """95-%-Intervall des Mittelwerts per Block-Bootstrap (berücksichtigt überlappende Labels)."""
    x = np.asarray(x, float)
    n = len(x)
    rng = np.random.default_rng(seed)
    nb = int(np.ceil(n / block))
    starts = np.arange(n - block + 1)
    means = np.empty(n_boot)
    for b in range(n_boot):
        s = rng.choice(starts, nb)
        means[b] = x[(s[:, None] + np.arange(block)).ravel()[:n]].mean()
    return np.percentile(means, [2.5, 97.5])


def mcnemar(model_ok, other_ok):
    """Exakter McNemar-Test (gepaarte Treffer, zweiseitig)."""
    oa, ob = int((model_ok & ~other_ok).sum()), int((~model_ok & other_ok).sum())
    return binomtest(oa, oa + ob, 0.5).pvalue if oa + ob else 1.0


def holm(p):
    """Holm-Bonferroni-adjustierte p-Werte."""
    p = np.asarray(p, float)
    m = len(p)
    adj = np.empty(m)
    run = 0.0
    for rank, idx in enumerate(np.argsort(p)):
        run = max(run, (m - rank) * p[idx])
        adj[idx] = min(1.0, run)
    return adj
'''
FILES['features'] = r'''import numpy as np
import pandas as pd

from .config import EXO


def build_features(df, H, exo=EXO):
    """Stationäre Merkmale (Log-Returns, Nachlauf-Lücke, Volatilität, Wochentag) für Serie H.

    Jedes Merkmal in Zeile t nutzt nur Preise bis einschließlich t.
    """
    df = df.reset_index(drop=True)
    F = pd.DataFrame({"date": df["date"]})
    lp = {c: np.log(df[c].where(df[c] > 0)) for c in [H] + list(exo)}
    for c, s in lp.items():
        r1 = s.diff()
        F[f"{c}_r1"] = r1
        F[f"{c}_r1_l1"] = r1.shift(1)
        F[f"{c}_r1_l2"] = r1.shift(2)
        F[f"{c}_r3"] = s - s.shift(3)
        F[f"{c}_r5"] = s - s.shift(5)
    for c in exo:                                   # Nachlauf-Lücke: Marktbewegung minus Preisbewegung
        F[f"gap1_{c}"] = F[f"{c}_r1"] - F[f"{H}_r1"]
        F[f"gap3_{c}"] = F[f"{c}_r3"] - F[f"{H}_r3"]
    r1h = F[f"{H}_r1"]
    F["vol5_H"] = r1h.rolling(5).std()
    F["vol20_H"] = r1h.rolling(20).std()
    F["dev20_H"] = lp[H] - lp[H].rolling(20).mean()
    F["weekday"] = df["date"].dt.weekday
    return F


def london_features(H, F):
    """Argus-only-Variante: eigene Serie + ICE Gasoil + EUR/USD."""
    keep = ["date", "weekday", "vol5_H", "vol20_H", "dev20_H"]
    for c in [H, "ice_gasoil", "usd_eur"]:
        keep += [f"{c}_r1", f"{c}_r1_l1", f"{c}_r1_l2", f"{c}_r3", f"{c}_r5"]
    for c in ["ice_gasoil", "usd_eur"]:
        keep += [f"gap1_{c}", f"gap3_{c}"]
    return F[keep]


def merkmal_beschreibung(name, H):
    """Verständliche Beschreibung eines Merkmals aus build_features (für die Darstellung im Notebook)."""
    from .config import EXO, SERIEN_NAMEN
    namen = {"ice_gasoil": "ICE Gasoil", "brent": "Brent", "wti": "WTI", "nymex_heating_oil": "NYMEX Heizöl",
             "usd_eur": "Dollar-Euro-Kurs"}
    namen[H] = f"Preis {SERIEN_NAMEN.get(H, H)}"
    spezial = {"vol5_H": "Schwankung des eigenen Preises (Standardabweichung der Tagesrenditen, 5 Tage)",
               "vol20_H": "Schwankung des eigenen Preises (Standardabweichung der Tagesrenditen, 20 Tage)",
               "dev20_H": "Abstand des eigenen Preises vom 20-Tage-Durchschnitt (log)",
               "weekday": "Wochentag (0 = Montag)"}
    if name in spezial:
        return spezial[name]
    for kind, text in (("gap1_", "Nachlauf-Lücke 1 Tag: Rendite {x} minus Rendite des eigenen Preises"),
                       ("gap3_", "Nachlauf-Lücke 3 Tage: Rendite {x} minus Rendite des eigenen Preises")):
        if name.startswith(kind):
            return text.format(x=namen.get(name[len(kind):], name[len(kind):]))
    for suffix, text in (("_r1_l1", "Tagesrendite von gestern"), ("_r1_l2", "Tagesrendite von vorgestern"),
                         ("_r1", "Tagesrendite heute"), ("_r3", "Rendite der letzten 3 Tage"),
                         ("_r5", "Rendite der letzten 5 Tage")):
        if name.endswith(suffix):
            return f"{namen.get(name[:-len(suffix)], name[:-len(suffix)])}: {text}"
    return name
'''
FILES['leakage'] = r'''import numpy as np

from .features import build_features


def leak_test(d, H, n_cut=1500, n_check=50):
    """Features dürfen sich nicht ändern, wenn man die Zukunft abschneidet.

    Braucht mindestens n_cut Zeilen. Wirft AssertionError bei Look-ahead.
    """
    if len(d) < n_cut:
        raise ValueError(f"leak_test braucht >= {n_cut} Zeilen, d hat {len(d)}")
    a = build_features(d.iloc[:n_cut], H).iloc[n_cut - n_check:n_cut].drop(columns="date").values
    b = build_features(d, H).iloc[n_cut - n_check:n_cut].drop(columns="date").values
    assert np.allclose(a, b, equal_nan=True), f"Look-ahead in den Features ({H})!"
'''
FILES['walk_forward'] = r'''import numpy as np
import pandas as pd
from xgboost import XGBClassifier

from .config import HORIZON, REG


def walk_forward(d, H, F, window, entry_lag=0, feat_lag=0, retrain_every=5, params=REG, horizon=HORIZON,
                 model_factory=None, label_shift=0, keep_unlabeled=False):
    """Gepurgter Walk-Forward (expanding window).

    d: Datentabelle (Spalte date + Preisspalten), F: Features zu d, window: (start, ende).
    Ziel: Preis(t+horizon) > Preis(t+entry_lag). Features: Information bis t-feat_lag.
    Training nur mit Zeilen, deren Label am Entscheidungstag i schon bekannt ist:
    last = i - horizon - feat_lag.
    model_factory: Funktion ohne Argumente, die ein frisches Modell (fit/predict_proba) liefert.
    Ohne Angabe: XGBClassifier(**params) wie in db_08.
    keep_unlabeled: True = auch Tage ohne bekanntes Ergebnis zurückgeben (y_true = NaN), z. B. die letzten 3 Tage.
    label_shift: Zufallstest (Placebo). Zeile j bekommt das Label von Zeile j - label_shift, die Beziehung
    zwischen Merkmalen und Ziel ist damit zerstört. 0 = echtes Ziel. Es werden nur frühere Labels verwendet.
    """
    feats = [c for c in F.columns if c != "date"]
    P = d[H].astype(float)
    p0, p1 = P.shift(-entry_lag), P.shift(-horizon)
    y = (p1 > p0).astype(float).where(p0.notna() & p1.notna())
    if label_shift:                                   # Placebo; Zeilen ohne echtes Label (Datenende) bleiben leer
        y = y.shift(label_shift).where(y.notna())
    X = F[feats].shift(feat_lag).values.astype(float)
    yv = y.values
    idx = np.where((d["date"] >= window[0]) & (d["date"] <= window[1]))[0]
    proba, model = [], None
    for k, i in enumerate(idx):
        if model is None or k % retrain_every == 0:
            last = i - horizon - feat_lag
            Xt, yt = X[:last + 1], yv[:last + 1]
            m = ~np.isnan(yt)
            model = (model_factory() if model_factory else XGBClassifier(**params)).fit(Xt[m], yt[m])
        proba.append(model.predict_proba(X[i:i + 1])[0, 1])
    res = pd.DataFrame({"date": d["date"].values[idx], "y_true": yv[idx], "proba": proba})
    if keep_unlabeled:
        return res.reset_index(drop=True)
    return res.dropna(subset=["y_true"]).reset_index(drop=True)
'''
FILES['evaluate'] = r'''import numpy as np
import pandas as pd
from scipy.stats import binomtest

from .config import HORIZON
from .stats import block_boot_mean, mcnemar, wil


def evaluate_run(d, H, F, res, entry_lag, feat_lag, horizon=HORIZON):
    """Pflichtmetriken für einen Walk-Forward-Lauf (Logik 1:1 aus db_08).

    F muss das volle Merkmalsset enthalten (für die Baselines NYMEX-Regel und Momentum).
    """
    y, p = res["y_true"].values, res["proba"].values
    pred = (p >= 0.5).astype(int)
    ok = (pred == y)
    n = len(res)
    hit = int(ok.sum())
    lo, hi = wil(hit, n)
    blo, bhi = block_boot_mean(ok.astype(float))
    p_non = max(binomtest(int(ok[o::3].sum()), len(ok[o::3]), 0.5).pvalue for o in range(3))
    base = y.mean()
    brier, ref = np.mean((p - y) ** 2), np.mean((base - y) ** 2)
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    # Vorzeichenregeln als Baselines (gleiche Tage)
    pos = pd.Series(np.arange(len(d)), index=d["date"])
    i = pos.reindex(res["date"]).values.astype(int)
    nym = (F["nymex_heating_oil_r1"].shift(feat_lag).values[i] > 0).astype(int)
    mom = (F[f"{H}_r1"].shift(feat_lag).values[i] > 0).astype(int)
    # Kosten (Regret): kaufen = Preis(t+entry_lag), warten = Preis(t+horizon)
    p_now, p_fut = d[H].values[i + entry_lag], d[H].values[i + horizon]
    best = np.minimum(p_now, p_fut)
    c_mod = np.where(pred == 1, p_now, p_fut) - best
    c_buy = p_now - best
    diff = c_buy - c_mod                                   # > 0: Modell billiger als immer kaufen
    dlo, dhi = block_boot_mean(diff)
    return {"Serie": H, "n": n, "DA": hit / n, "Wilson_lo": lo, "Wilson_hi": hi,
            "Boot_lo": blo, "Boot_hi": bhi, "p_nicht_ueberl": p_non,
            "Mehrheit": max(base, 1 - base), "TP": tp, "FP": fp, "TN": tn, "FN": fn,
            "Precision": tp / max(tp + fp, 1), "Recall": tp / max(tp + fn, 1),
            "Brier_Skill": 1 - brier / ref,
            "DA_NYMEX_Regel": float((nym == y).mean()), "DA_Momentum": float((mom == y).mean()),
            "p_McNemar_NYMEX": mcnemar(ok, nym == y), "p_McNemar_Momentum": mcnemar(ok, mom == y),
            "Regret_Modell": c_mod.mean(), "Regret_immer_kaufen": c_buy.mean(),
            "Ersparnis_vs_kaufen": diff.mean(), "Ersp_lo": dlo, "Ersp_hi": dhi,
            "davon_zu_frueh_gekauft": c_mod[pred == 1].sum() / n,
            "davon_zu_lange_gewartet": c_mod[pred == 0].sum() / n}
'''
FILES['data'] = r'''"""Daten aus den Excel-Dateien laden und zu einer Tabelle `d` zusammenführen.

Argus-Excel (Add-in): Blatt "Tabelle1", Zeile 2 = Beschreibung, ab Zeile 3 Tageswerte.
Marktdaten-Excel: je Blatt Datum + Wert (Brent, WTI, Heizöl/NYMEX, Wechselkurs).
"""
import warnings

import numpy as np
import pandas as pd

from .config import EXO, SERIES

_REGION = [("southwest", "suedwest"), ("southeast", "suedost"), ("rhine-main", "rheinmain"), ("south", "sued")]


def _serie_name(desc):
    """Beschreibungstext der Argus-Spalte -> Serienname (oder None, z. B. 'all regions')."""
    t = str(desc).lower()
    if "ice nwe month 1" in t:
        return "ice_gasoil"
    if "all regions" in t:
        return None
    produkt = "diesel" if "diesel en 590" in t else "heizoel" if "heating oil" in t else None
    if produkt is None:
        return None
    for key, name in _REGION:
        if key in t:
            return f"{produkt}_{name}"
    return None


def load_argus_excel(path, sheet="Tabelle1", return_info=False):
    """Argus-Excel laden. Mit return_info=True zusätzlich eine Tabelle: welche Spalte wurde welcher Serie zugeordnet."""
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    head = raw.iloc[1]                                   # Zeile 2: Beschreibungen
    body = raw.iloc[2:].copy()
    cols = {0: "date"}
    for j in range(1, raw.shape[1]):
        n = _serie_name(head.iloc[j])
        if n:
            cols[j] = n
    df = body[list(cols)].rename(columns=cols)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"]).sort_values("date")
    for c in df.columns[1:]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    fehlt = [c for c in SERIES + ["ice_gasoil"] if c not in df.columns]
    if fehlt:
        raise ValueError(f"Argus-Spalten nicht erkannt: {fehlt}")
    df = df.drop_duplicates("date", keep="last").reset_index(drop=True)
    if not return_info:
        return df
    rows = []
    for j, n in cols.items():
        if n == "date":
            continue
        v = df[n]
        rows.append({"Serie": n, "Excel-Spalte": j + 1, "Beschreibung": str(head.iloc[j])[:110],
                     "Tage": int(v.notna().sum()), "fehlend": int(v.isna().sum()),
                     "erster": df.loc[v.first_valid_index(), "date"].date(),
                     "letzter": df.loc[v.last_valid_index(), "date"].date(),
                     "Min": round(float(v.min()), 3), "Max": round(float(v.max()), 3),
                     "letzter Wert": round(float(v.dropna().iloc[-1]), 3)})
    return df, pd.DataFrame(rows)


def inspect_excel(path, n=6):
    """Struktur jeder Tabelle ausgeben (Blattnamen, Größe, erste Zeilen) – zum Prüfen vor dem Laden."""
    xl = pd.ExcelFile(path)
    for s in xl.sheet_names:
        df = xl.parse(s, header=None, nrows=n + 4)
        print(f"--- Blatt '{s}' (erste Zeilen) ---")
        print(df.head(n + 4).to_string(max_colwidth=40))


def load_market_sheet(path, sheet, name, date_col=None, value_col=None, header_row=None, return_info=False):
    """Ein Marktdaten-Blatt als Serie `name`.

    Ohne Angabe wird das Blatt automatisch gelesen: erste Spalte mit Datumswerten, danach die
    erste Spalte mit überwiegend Zahlen. Bei Zweifel date_col/value_col (0-basiert) und
    header_row angeben, nachdem `inspect_excel` die Struktur gezeigt hat.
    """
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        dt = raw.apply(lambda s: pd.to_datetime(s, errors="coerce"))
    if date_col is None:
        is_date = [(dt[c].notna() & pd.to_numeric(raw[c], errors="coerce").isna()).mean() for c in raw.columns]
        date_col = int(np.argmax(is_date))
        if max(is_date) < 0.5:
            raise ValueError(f"Blatt '{sheet}': keine Datumsspalte gefunden")
    if value_col is None:
        num = [(pd.to_numeric(raw[c], errors="coerce").notna().mean() if c != date_col else -1) for c in raw.columns]
        value_col = int(np.argmax(num))
    out = pd.DataFrame({"date": dt[date_col], name: pd.to_numeric(raw[value_col], errors="coerce")})
    out = out.dropna(subset=["date", name])
    out = out[out["date"] >= "1900-01-01"].drop_duplicates("date", keep="last").sort_values("date")
    first_row = out.index.min()
    out = out.reset_index(drop=True)
    if not return_info:
        return out
    def kopf(c):                                           # Textzellen oberhalb der ersten Daten (z. B. Reihenname)
        txt = [str(x)[:40] for x in raw.loc[: first_row - 1, c] if isinstance(x, str)]
        return " | ".join(txt[-3:])
    info = {"Variable": name, "Blatt": sheet, "Datumsspalte": date_col + 1, "Wertspalte": value_col + 1,
            "Kopfzeile Wert": kopf(value_col), "Tage": len(out),
            "erster": out["date"].min().date(), "letzter": out["date"].max().date(),
            "Min": round(float(out[name].min()), 4), "Max": round(float(out[name].max()), 4),
            "letzter Wert": round(float(out[name].iloc[-1]), 4)}
    return out, info


def build_dataset(argus, markt, max_ffill=5):
    """Argus-Handelstage als Gerüst, Marktdaten per Datum dazu, Lücken bis `max_ffill` Tage vorwärts füllen.

    Rückgabe: (d, bericht). Das Vorwärtsfüllen nutzt nur vergangene Werte (kein Look-ahead).
    """
    d = argus.copy()
    for m in markt:
        d = d.merge(m, on="date", how="left")
    cols = SERIES + EXO
    fehlt = [c for c in cols if c not in d.columns]
    if fehlt:
        raise ValueError(f"Spalten fehlen: {fehlt}")
    vorher = d[cols].isna().sum()
    d[cols] = d[cols].ffill(limit=max_ffill)
    d = d.dropna(subset=cols).reset_index(drop=True)
    bericht = pd.DataFrame({"fehlend_vor_ffill": vorher, "fehlend_nach_ffill": d[cols].isna().sum()})
    return d, bericht
'''
FILES['models'] = r'''"""Modelle für den Vergleich (db_10). Alle bekommen dieselben Daten, denselben Walk-Forward.

Imputation (Median) und Skalierung sind Teil des Modells und werden bei jedem Training nur auf den
Trainingsdaten gelernt (kein Look-ahead).
"""
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from .config import REG


def make_xgb():
    return XGBClassifier(**REG)


def make_logreg():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(C=0.05, max_iter=1000))


def make_rf():
    return make_pipeline(SimpleImputer(strategy="median"),
                         RandomForestClassifier(n_estimators=200, max_depth=4, min_samples_leaf=20,
                                                max_features="sqrt", random_state=42, n_jobs=-1))


def make_majority():
    return DummyClassifier(strategy="prior")          # immer die im Training häufigere Klasse


def klein_features(H):
    """Kleines Modell: nur die wichtigsten Tagesrenditen und Nachlauf-Lücken."""
    return [f"{H}_r1", "ice_gasoil_r1", "nymex_heating_oil_r1", "brent_r1", "wti_r1",
            "gap1_ice_gasoil", "gap1_nymex_heating_oil"]


# Name -> (Fabrik, Merkmale: None = alle 44, "klein" = kleine Auswahl)
MODELLE = {
    "XGBoost": (make_xgb, None),
    "Logistische Regression": (make_logreg, None),
    "Random Forest": (make_rf, None),
    "Logistisch klein (7 Merkmale)": (make_logreg, "klein"),
    "Mehrheitsklasse": (make_majority, None),
}
'''
FILES['beobachten'] = r'''"""Dreistufiges Signal: Kaufen / Warten / Beobachten.

Regel (vorab festgelegt):
    Kaufen      wenn p >= 0.5 + delta
    Warten      wenn p <  0.5 - delta
    Beobachten  sonst (Modell ist sich nicht sicher, keine Entscheidung)
delta = 0 entspricht dem bisherigen Zwei-Stufen-Signal (nie Beobachten).

Kostenannahme: Beobachten wird einmal wie "Warten" (Variante A) und einmal wie "Kaufen" (Variante B) gerechnet.
"""
import numpy as np
import pandas as pd

from .config import HORIZON
from .stats import block_boot_mean, wil

KAUFEN, WARTEN, BEOBACHTEN = 1, -1, 0
NAMEN = {KAUFEN: "Kaufen", WARTEN: "Warten", BEOBACHTEN: "Beobachten"}


def signal(proba, delta):
    p = np.asarray(proba, float)
    return np.where(p >= 0.5 + delta, KAUFEN, np.where(p < 0.5 - delta, WARTEN, BEOBACHTEN))


def bewerte_dreistufig(d, H, res, delta, entry_lag=0, horizon=HORIZON):
    """Kennzahlen eines dreistufigen Signals. res: Spalten date, y_true, proba (wie walk_forward)."""
    y, p = res["y_true"].values, res["proba"].values
    sig = signal(p, delta)
    n = len(res)
    kauf, wart = sig == KAUFEN, sig == WARTEN
    entsch = kauf | wart
    ok = np.where(kauf, y == 1, np.where(wart, y == 0, False))
    n_e = int(entsch.sum())
    out = {"Serie": H, "delta": delta, "n": n, "n_Kaufen": int(kauf.sum()), "n_Warten": int(wart.sum()),
           "n_Beobachten": int((~entsch).sum()), "Abdeckung": n_e / n}
    if n_e:
        hit = int(ok[entsch].sum())
        lo, hi = wil(hit, n_e)
        out.update({"DA_entschieden": hit / n_e, "Wilson_lo": lo, "Wilson_hi": hi})
        if n_e >= 20:
            blo, bhi = block_boot_mean(ok[entsch].astype(float))
            out.update({"Boot_lo": blo, "Boot_hi": bhi})
    out.setdefault("Boot_lo", np.nan); out.setdefault("Boot_hi", np.nan)
    out["Precision_Kaufen"] = float((y[kauf] == 1).mean()) if kauf.any() else np.nan
    out["Precision_Warten"] = float((y[wart] == 0).mean()) if wart.any() else np.nan
    out["Basisrate"] = float(y.mean())
    # Kosten wie in evaluate_run: kaufen = Preis(t+entry_lag), warten = Preis(t+horizon)
    pos = pd.Series(np.arange(len(d)), index=d["date"])
    i = pos.reindex(res["date"]).values.astype(int)
    p_now, p_fut = d[H].values[i + entry_lag], d[H].values[i + horizon]
    best = np.minimum(p_now, p_fut)
    c_alle = p_now - best
    c_a = np.where(kauf, p_now, p_fut) - best                 # Beobachten = Warten
    c_b = np.where(wart, p_fut, p_now) - best                 # Beobachten = Kaufen
    for name, c in (("A", c_a), ("B", c_b)):
        diff = c_alle - c
        lo, hi = block_boot_mean(diff)
        out.update({f"Ersparnis_{name}": float(diff.mean()), f"Ersp_{name}_lo": lo, f"Ersp_{name}_hi": hi})
    return out
'''
FILES['plots'] = r'''"""Grafiken für das Gesamt-Notebook. Jede Funktion gibt eine matplotlib-Figure zurück (Aufruf: plt.show())."""
import numpy as np
import matplotlib.pyplot as plt

from .config import SERIEN_NAMEN
from .stats import wil

SURF, INK, INK2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#8a8984", "#e6e5e1"
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#8a8984"
DUNKEL, HELL = "#256abf", "#9ec5f4"


def _stil():
    plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
                         "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
                         "axes.edgecolor": GRID})


def _kopf(fig, titel, sub=None):
    fig.text(0.04, 0.96, titel, fontsize=15, fontweight="bold", ha="left", va="top")
    if sub:
        fig.text(0.04, 0.905, sub, fontsize=9.5, color=INK2, ha="left", va="top")


def _kommas(x, nd=1):
    return f"{x:.{nd}f}".replace(".", ",")


def _name(s):
    return SERIEN_NAMEN.get(s, s)


def _achsen(ax):
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.tick_params(length=0)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)


def konfusion_einzeln(ax, tp, fp, fn, tn, titel=None, gross=True):
    """Eine Konfusionsmatrix: Zeilen = Modell Kaufen/Warten, Spalten = Preis stieg / stieg nicht."""
    zellen = [[tp, fp], [fn, tn]]
    labels = [["richtig gekauft", "zu früh gekauft"], ["zu lange gewartet", "richtig gewartet"]]
    for i in range(2):
        for j in range(2):
            dunkel = i == j
            ax.add_patch(plt.Rectangle((j + 0.03, 1 - i + 0.03), 0.94, 0.94, color=DUNKEL if dunkel else HELL, zorder=1))
            ax.text(j + 0.5, 1 - i + (0.6 if gross else 0.55), str(int(zellen[i][j])), ha="center", va="center",
                    fontsize=22 if gross else 13, fontweight="bold", color="#ffffff" if dunkel else INK, zorder=2)
            ax.text(j + 0.5, 1 - i + (0.28 if gross else 0.22), labels[i][j], ha="center", va="center",
                    fontsize=9.5 if gross else 6.5, color="#ffffff" if dunkel else INK, zorder=2)
    ax.set_xlim(0, 2); ax.set_ylim(0, 2)
    ax.set_xticks([0.5, 1.5]); ax.set_xticklabels(["Preis stieg", "stieg nicht"], fontsize=9 if gross else 7)
    ax.set_yticks([0.5, 1.5]); ax.set_yticklabels(["Warten", "Kaufen"], fontsize=9 if gross else 7)
    ax.xaxis.tick_top(); ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    if titel:
        ax.set_title(titel, fontsize=10 if gross else 8.5, color=INK, pad=14 if gross else 12)


def konfusion_raster(df, titel="Fehlerübersicht aller Serien (Konfusionsmatrizen)", sub=None):
    """df: Spalten Serie, TP, FP, FN, TN, DA. Zeigt alle Serien als Raster."""
    _stil()
    n = len(df)
    spalten = 4
    zeilen = int(np.ceil(n / spalten))
    fig, axs = plt.subplots(zeilen, spalten, figsize=(12, 3.1 * zeilen + 1.0), dpi=110)
    fig.subplots_adjust(left=0.07, right=0.97, top=0.78 if zeilen == 2 else 0.7, bottom=0.04, hspace=0.55, wspace=0.35)
    axs = np.atleast_1d(axs).ravel()
    for ax, (_, r) in zip(axs, df.iterrows()):
        konfusion_einzeln(ax, r["TP"], r["FP"], r["FN"], r["TN"], f"{_name(r['Serie'])}: {_kommas(r['DA']*100)} % richtig", gross=False)
    for ax in axs[n:]:
        ax.axis("off")
    _kopf(fig, titel, sub or "Dunkelblau = richtig, hellblau = Fehler. Zeilen: was das Modell sagt. Spalten: was der Preis wirklich tat.")
    return fig


def trefferquote_mit_baselines(df, titel="Trefferquote je Serie", sub=None):
    """df: Serie, DA, Wilson_lo, Wilson_hi, Mehrheit, DA_NYMEX_Regel."""
    _stil()
    fig, ax = plt.subplots(figsize=(8, 4.8), dpi=110)
    fig.subplots_adjust(left=0.2, right=0.96, top=0.76, bottom=0.26)
    _achsen(ax)
    y = np.arange(len(df))[::-1]
    ax.axvline(0.5, color=INK2, lw=1, ls=(0, (3, 3)), zorder=1)
    ax.hlines(y, df["Wilson_lo"], df["Wilson_hi"], color=BLUE, lw=3, zorder=2, capstyle="round")
    ax.scatter(df["DA"], y, s=70, color=BLUE, zorder=4, edgecolor=SURF, linewidth=1.5, label="Modell (XGBoost)")
    ax.scatter(df["DA_NYMEX_Regel"], y, s=42, marker="D", color=ORANGE, zorder=3, edgecolor=SURF, linewidth=1, label="NYMEX-Regel (einfache Regel)")
    ax.scatter(df["Mehrheit"], y, s=42, marker="s", color=GREY, zorder=3, edgecolor=SURF, linewidth=1, label="Immer häufigere Richtung tippen")
    for yi, v in zip(y, df["DA"]):
        ax.text(v, yi + 0.33, f"{_kommas(v*100)} %", ha="center", fontsize=8, color=INK)
    ax.set_yticks(y); ax.set_yticklabels([_name(s) for s in df["Serie"]], fontsize=9.5)
    ax.set_ylim(-0.7, y.max() + 0.6)
    ax.text(0.502, -0.55, "50 % = Münzwurf", fontsize=8, color=INK2)
    lo = min(0.45, float(df["Wilson_lo"].min()) - 0.02)
    ax.set_xlim(lo, 0.75)
    ticks = [t for t in (0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75) if t >= lo]
    ax.set_xticks(ticks); ax.set_xticklabels([f"{int(round(t*100))} %" for t in ticks])
    ax.legend(loc="lower center", bbox_to_anchor=(0.45, -0.24), ncol=3, frameon=False, fontsize=8.5, labelcolor=INK2)
    _kopf(fig, titel, sub or "Balken = 95-%-Konfidenzintervall (Wilson). Bei überlappenden 3-Tage-Labels nur Orientierung.")
    return fig


def precision_recall(df, titel="Precision, Recall und Basisrate", sub=None):
    """df: Serie, Precision, Recall, Basisrate."""
    _stil()
    fig, ax = plt.subplots(figsize=(8, 4.8), dpi=110)
    fig.subplots_adjust(left=0.2, right=0.96, top=0.76, bottom=0.26)
    _achsen(ax)
    y = np.arange(len(df))[::-1]
    ax.scatter(df["Basisrate"], y, s=60, color=GREY, zorder=3, edgecolor=SURF, linewidth=1.5, label="Basisrate (beliebiger Tag)")
    ax.scatter(df["Precision"], y, s=70, color=BLUE, zorder=3, edgecolor=SURF, linewidth=1.5, label="Precision (Kaufsignal stimmt)")
    ax.scatter(df["Recall"], y, s=60, marker="D", color=ORANGE, zorder=3, edgecolor=SURF, linewidth=1, label="Recall (Anstieg erkannt)")
    ax.set_yticks(y); ax.set_yticklabels([_name(s) for s in df["Serie"]], fontsize=9.5)
    lo = min(0.4, float(df[["Precision", "Recall", "Basisrate"]].min().min()) - 0.02)
    hi = max(0.75, float(df[["Precision", "Recall", "Basisrate"]].max().max()) + 0.02)
    ax.set_xlim(lo, hi)
    ticks = [t for t in np.arange(0.4, 0.81, 0.05) if lo <= t <= hi]
    ax.set_xticks(ticks); ax.set_xticklabels([f"{int(round(t*100))} %" for t in ticks])
    ax.legend(loc="lower center", bbox_to_anchor=(0.45, -0.24), ncol=3, frameon=False, fontsize=8.5, labelcolor=INK2)
    _kopf(fig, titel, sub or "Precision = von allen Kaufsignalen stieg der Preis wirklich. Recall = von allen Anstiegen erkannt.")
    return fig


def kalibrierung(p, y, titel="Wie verlässlich ist die Wahrscheinlichkeit des Modells?", sub=None):
    """Tatsächlicher Anteil steigender Preise je Bereich der vorhergesagten Wahrscheinlichkeit (alle Serien gepoolt)."""
    _stil()
    p, y = np.asarray(p, float), np.asarray(y, float)
    kanten = np.array([0.0, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 1.0])
    mitten, raten, los, his, ns = [], [], [], [], []
    for a, b in zip(kanten[:-1], kanten[1:]):
        m = (p >= a) & (p < b)
        if m.sum() >= 20:
            k = int(y[m].sum()); lo, hi = wil(k, int(m.sum()))
            mitten.append(p[m].mean()); raten.append(y[m].mean()); los.append(lo); his.append(hi); ns.append(int(m.sum()))
    fig, ax = plt.subplots(figsize=(7, 5.2), dpi=110)
    fig.subplots_adjust(left=0.12, right=0.96, top=0.78, bottom=0.14)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
    ax.plot([0.25, 0.75], [0.25, 0.75], color=MUTED, lw=1, ls=(0, (3, 3)), label="perfekt verlässlich")
    ax.errorbar(mitten, raten, yerr=[np.array(raten) - np.array(los), np.array(his) - np.array(raten)], fmt="o", color=BLUE,
                ms=8, capsize=3, lw=2, label="Modell (Punkt = Bereich, Balken = 95-%-Intervall)")
    for x, n in zip(mitten, ns):
        ax.text(x, 0.205, f"n={n}", ha="center", fontsize=7.5, color=INK2)
    ax.set_xlim(0.25, 0.75); ax.set_ylim(0.18, 0.78)
    ax.set_xlabel("Wahrscheinlichkeit laut Modell, dass der Preis in 3 Tagen höher ist")
    ax.set_ylabel("Tatsächlicher Anteil steigender Preise")
    ax.set_xticks(np.arange(0.3, 0.76, 0.1)); ax.set_xticklabels([f"{int(round(t*100))} %" for t in np.arange(0.3, 0.76, 0.1)])
    ax.set_yticks(np.arange(0.3, 0.76, 0.1)); ax.set_yticklabels([f"{int(round(t*100))} %" for t in np.arange(0.3, 0.76, 0.1)])
    ax.legend(loc="upper left", frameon=False, fontsize=8.5, labelcolor=INK2)
    _kopf(fig, titel, sub or "Je höher die Wahrscheinlichkeit, desto öfter sollte der Preis wirklich steigen (Punkte nahe der gestrichelten Linie).")
    return fig


def abdeckung_kurve(df, titel="Weniger Signale, dafür zuverlässiger", sub=None):
    """df: Fenster, Abdeckung, DA_entschieden (eine Zeile je Anteil und Zeitraum)."""
    _stil()
    fig, ax = plt.subplots(figsize=(8, 4.8), dpi=110)
    fig.subplots_adjust(left=0.12, right=0.96, top=0.76, bottom=0.2)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
    farben = {"Entwicklung": BLUE, "Test": ORANGE}
    for f, g in df.groupby("Fenster"):
        g = g.sort_values("Abdeckung")
        ax.plot(g["Abdeckung"] * 100, g["DA_entschieden"] * 100, "-o", color=farben.get(f, GREY), lw=2, ms=6, label=f)
    ax.axhline(50, color=INK2, lw=1, ls=(0, (3, 3)))
    ax.annotate("50 % = Münzwurf", xy=(1, 50), xycoords=("axes fraction", "data"), xytext=(-4, 4), textcoords="offset points",
                ha="right", fontsize=8, color=INK2)
    ax.set_ylim(48, max(66, float(df["DA_entschieden"].max() * 100) + 2))
    ax.set_xlabel("Anteil der Tage mit Signal Kaufen oder Warten (Rest = Beobachten)")
    ax.set_ylabel("Trefferquote auf diesen Tagen")
    ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0f} %"); ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0f} %")
    ax.invert_xaxis()
    ax.legend(loc="upper right", frameon=False, fontsize=8.5, labelcolor=INK2)
    _kopf(fig, titel, sub or "Nach links: mehr Tage werden zu „Beobachten“. Mittel über alle Serien.")
    return fig


def ersparnis(df, titel="Ersparnis gegenüber „immer kaufen“", sub=None):
    """df: Serie, Ersparnis_vs_kaufen, Ersp_lo, Ersp_hi (Euro pro 100 l und Tag)."""
    _stil()
    fig, ax = plt.subplots(figsize=(8, 4.8), dpi=110)
    fig.subplots_adjust(left=0.2, right=0.96, top=0.76, bottom=0.14)
    _achsen(ax)
    y = np.arange(len(df))[::-1]
    ax.axvline(0, color=INK, lw=1.2, zorder=1)
    ax.hlines(y, df["Ersp_lo"], df["Ersp_hi"], color=BLUE, lw=3, zorder=2, capstyle="round")
    ax.scatter(df["Ersparnis_vs_kaufen"], y, s=70, color=BLUE, zorder=4, edgecolor=SURF, linewidth=1.5)
    for yi, v, lo in zip(y, df["Ersparnis_vs_kaufen"], df["Ersp_lo"]):
        ax.text(v, yi + 0.33, f"{_kommas(v, 2)} €", ha="center", fontsize=8, color=INK)
        if lo < 0:
            ax.text(df["Ersp_hi"].max() + 0.02, yi, "Intervall reicht unter 0", fontsize=7.5, color=INK2, va="center")
    ax.set_yticks(y); ax.set_yticklabels([_name(s) for s in df["Serie"]], fontsize=9.5)
    ax.set_xlim(min(-0.1, float(df["Ersp_lo"].min()) - 0.02), float(df["Ersp_hi"].max()) + 0.22)
    ticks = [t for t in np.arange(-0.2, 1.0, 0.1) if ax.get_xlim()[0] <= t <= df["Ersp_hi"].max() + 0.05]
    ax.set_xticks(ticks); ax.set_xticklabels([f"{_kommas(t)} €" for t in ticks])
    _kopf(fig, titel, sub or "Euro pro 100 Liter und Tag. Rechts von der Null = Modell ist günstiger. Balken = 95-%-Bootstrap-Intervall.")
    return fig
'''
FILES['neue_daten'] = r'''"""Neue, bisher ungesehene Tage: Daten laden, mit der Historie verbinden, Signale erzeugen."""
import glob
import os

import numpy as np
import pandas as pd

from .beobachten import NAMEN, signal
from .config import SERIES
from .data import load_argus_excel
from .features import build_features
from .walk_forward import walk_forward

SPALTEN = SERIES + ["ice_gasoil"]


def suche_neueste_api_csv(ordner):
    """Neueste Datei argus_api_*.csv im Ordner (nach Namen sortiert) oder None."""
    treffer = sorted(glob.glob(os.path.join(ordner, "argus_api_*.csv")))
    return treffer[-1] if treffer else None


def lade_neue_argus(pfad):
    """CSV (Spalte date + 8 Serien + ice_gasoil) oder Argus-Excel im Add-in-Format."""
    if str(pfad).lower().endswith(".csv"):
        df = pd.read_csv(pfad)
        datum = next((c for c in df.columns if c.lower() in ("date", "datum", "publicationdate")), df.columns[0])
        df["date"] = pd.to_datetime(df[datum].astype(str).str[:10])
        fehlt = [c for c in SPALTEN if c not in df.columns]
        if fehlt:
            raise ValueError(f"In der Datei fehlen Spalten: {fehlt}")
        df = df[["date"] + SPALTEN].copy()
        for c in SPALTEN:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        return df.sort_values("date").drop_duplicates("date", keep="last").reset_index(drop=True)
    return load_argus_excel(pfad)


def kombiniere_argus(alt, neu):
    """Historie + neue Tage. Bei Überlappung gelten die alten (geprüften) Werte; Abweichungen werden berichtet."""
    gemeinsam = alt["date"][alt["date"].isin(neu["date"])]
    a = alt.set_index("date").loc[gemeinsam, SPALTEN]
    n = neu.set_index("date").loc[gemeinsam, SPALTEN]
    abw = (a - n).abs().max().max() if len(gemeinsam) else np.nan
    zusatz = neu[~neu["date"].isin(alt["date"])]
    out = pd.concat([alt, zusatz]).sort_values("date").reset_index(drop=True)
    bericht = {"Tage_alt": len(alt), "Tage_ueberlappung": len(gemeinsam), "max_Abweichung_Ueberlappung": float(abw) if abw == abw else np.nan,
               "Tage_neu_dazu": len(zusatz), "erster_neuer_Tag": zusatz["date"].min() if len(zusatz) else None,
               "letzter_Tag": out["date"].max()}
    return out, bericht


def signale_neue_tage(d, H, start, ende, delta, retrain_every=5):
    """Vorhersage und Signal für jeden Tag von start bis ende, auch für Tage, deren Ergebnis noch nicht feststeht."""
    F = build_features(d, H)
    res = walk_forward(d, H, F, (start, ende), entry_lag=0, feat_lag=0, retrain_every=retrain_every, keep_unlabeled=True)
    res["Signal"] = [NAMEN[int(s)] for s in signal(res["proba"].values, delta)]
    res["Ergebnis_bekannt"] = res["y_true"].notna()
    res.insert(0, "Serie", H)
    return res, F
'''
for name, text in FILES.items():
    open(f"/tmp/p229_code/p229/{name}.py", "w").write(text)
print("Code angelegt:", sorted(os.listdir("/tmp/p229_code/p229")))

In [ ]:
# Setup: Pfade, Code, Daten laden. (Die Zelle davor legt den Projektcode an.)
import os, sys, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

BASE_PATH = "/Volumes/dev_workspace/p229/p229_files"
OUT = f"{BASE_PATH}/output"
GRAFIK = f"{OUT}/grafiken_gesamt"
os.makedirs(GRAFIK, exist_ok=True)
ARGUS_XLSX = f"{BASE_PATH}/Heizoel und Diesel bis 2016 und ICE Daten.xlsx"
MARKT_XLSX = f"{BASE_PATH}/P229 - Daten von Brent, WTI, Heizöl, Wechselkurs.xlsx"

sys.path.insert(0, "/tmp/p229_code")
from p229.config import SERIES, EXO, DEV, TEST, LAST_ARGUS, REG, SERIEN_NAMEN
from p229.data import load_argus_excel, load_market_sheet, build_dataset, inspect_excel
from p229.features import build_features, merkmal_beschreibung
from p229.leakage import leak_test
from p229.walk_forward import walk_forward
from p229.evaluate import evaluate_run
from p229.stats import holm
from p229.beobachten import bewerte_dreistufig, signal, NAMEN
from p229.neue_daten import suche_neueste_api_csv, lade_neue_argus, kombiniere_argus, signale_neue_tage
from p229 import plots

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 90)

def zeige(fig, name):
    """Grafik anzeigen und als PNG speichern."""
    fig.savefig(f"{GRAFIK}/{name}", dpi=200)
    plt.show()
    plt.close(fig)

argus, argus_info = load_argus_excel(ARGUS_XLSX, return_info=True)
BLAETTER = [("Brent Rohöl", "brent"), ("WIT Rohöl", "wti"), ("Heizöl", "nymex_heating_oil"), ("USD_EUR Wechselkurs", "usd_eur")]
markt, markt_info = [], []
for blatt, name in BLAETTER:
    m, i = load_market_sheet(MARKT_XLSX, blatt, name, return_info=True)
    markt.append(m); markt_info.append(i)
d_all, fuell_bericht = build_dataset(argus, markt)
d = d_all[d_all["date"] <= LAST_ARGUS].reset_index(drop=True)
for H in SERIES:
    leak_test(d, H)
print(f"Datentabelle: {d.shape[0]} Tage von {d['date'].min().date()} bis {d['date'].max().date()} | Leakage-Test bestanden")

## Teil A: Welche Daten nutzt das Modell?

In [ ]:
# A1: Quellen. Welche Datei, welches Blatt, welche Spalte?
print("=== 1. Preise aus der Argus-Excel (Blatt 'Tabelle1') ===")
print("Argus O.M.R., prompt, London close, Index, Euro/100 l, fca Lkw. ICE Gasoil: Futures-Abrechnungspreis, USD/t.")
print(argus_info.drop(columns="Beschreibung").to_string(index=False))
print("\n=== 2. Marktdaten aus der Marktdaten-Excel ===")
print(pd.DataFrame(markt_info).drop(columns=["Datumsspalte"]).to_string(index=False))
print("\n=== Lücken (Feiertage u. ä.) und wie sie behandelt werden ===")
print("Fehlende Werte werden mit dem letzten bekannten Wert gefüllt (höchstens 5 Tage). Das nutzt nur Vergangenheit.")
print(fuell_bericht.to_string())
print("\nNicht genutzt: Nachrichten, Rheinfracht, Argus 'all regions', low/high-Werte, Platts, OMR-Übersicht.")

In [ ]:
# A2: Die Eingabespalten des Modells (Merkmale), hier am Beispiel Heizöl Süd.
H0 = "heizoel_sued"
F0 = build_features(d, H0)
merkmale = [c for c in F0.columns if c != "date"]
tab = pd.DataFrame({"Nr": range(1, len(merkmale) + 1), "Merkmal (Spaltenname)": merkmale,
                    "Bedeutung": [merkmal_beschreibung(c, H0) for c in merkmale]})
print(f"Das Modell bekommt {len(merkmale)} Zahlen pro Tag. Alle sind Veränderungen (Renditen, Abstände, Schwankung), keine Preisniveaus.")
print(f"Beispiel für {SERIEN_NAMEN[H0]}; für die anderen Serien gilt dasselbe Muster mit dem jeweils eigenen Preis.\n")
print(tab.to_string(index=False))
print("\n=== Zielgröße (was vorhergesagt wird) ===")
print("Ziel = 1, wenn Preis(t+3 Handelstage) > Preis(t), sonst 0. Gleichstand zählt als 'nicht gestiegen'.")
print("\n=== Was das Modell NICHT sieht ===")
print("Preisniveaus, Monat/Jahreszeit, Nachrichten, Rheinfracht, Wetter, Zukunftsdaten, das Ergebnis selbst.")

In [ ]:
# A3: Beispielzeilen und Wichtigkeit der Merkmale.
from xgboost import XGBClassifier
print("=== So sehen die Eingaben der letzten 5 Tage aus (Heizöl Süd) ===")
print(F0.set_index("date")[merkmale].tail(5).T.round(4).to_string())

# Wichtigkeit: ein Modell mit allen Daten bis zum Testbeginn (nur zur Darstellung)
ziel = (d[H0].shift(-3) > d[H0]).astype(float).where(d[H0].shift(-3).notna())
i_test = int(np.where(d["date"] >= TEST[0])[0][0])
letzte = i_test - 3
X0 = F0.loc[:letzte, merkmale]; y0 = ziel.loc[:letzte]; m0 = y0.notna()
modell0 = XGBClassifier(**REG).fit(X0[m0], y0[m0])
gain = pd.Series(modell0.get_booster().get_score(importance_type="gain")).sort_values(ascending=False)
anteil = (gain / gain.sum() * 100).head(12)
plots._stil()
fig, ax = plt.subplots(figsize=(9, 7), dpi=110)
fig.subplots_adjust(left=0.42, right=0.95, top=0.82, bottom=0.06)
yy = np.arange(len(anteil))[::-1]
ax.barh(yy, anteil.values, color=plots.BLUE, height=0.6)
import textwrap
ax.set_yticks(yy); ax.set_yticklabels([textwrap.fill(merkmal_beschreibung(k, H0), 46) for k in anteil.index], fontsize=7.5)
for yi, v in zip(yy, anteil.values):
    ax.text(v + 0.3, yi, f"{v:.1f} %".replace(".", ","), va="center", fontsize=8)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.tick_params(length=0); ax.xaxis.set_visible(False)
plots._kopf(fig, "Welche Eingaben nutzt das Modell am stärksten?", f"Anteil am Gewinn je Eingabe, {SERIEN_NAMEN[H0]}, Modell mit Daten bis {d['date'].iloc[letzte].date()}. Zeigt Nutzung, keine Ursache.")
zeige(fig, "A_wichtigkeit.png")

## Teil B: Wie wurde getestet?

- **Walk-Forward:** Für jeden Tag im Testzeitraum wird das Modell nur mit Daten trainiert, deren Ergebnis an diesem Tag **schon bekannt** ist (alle Tage bis t − 3). Danach sagt es den Tag voraus. Alle 5 Tage wird neu trainiert. Es wird nichts aus der Zukunft benutzt.
- **Zeiträume:** Entwicklung 2023-04-21 bis 2024-11-01, Test 2024-11-15 bis 2026-01-20. Beide wurden vorher nie zum Anpassen des Modells benutzt, nur zur Auswahl von Regeln (siehe Teil D).
- **Modell:** XGBoost mit festen, stark vorsichtigen Einstellungen (keine Feinabstimmung).
- **Szenario S0:** Entscheidung nach Marktschluss am Tag t, Kauf zum Argus-Preis von Tag t.
- **Kontrollen:** Leakage-Test (Merkmale ändern sich nicht, wenn man die Zukunft abschneidet), Zufallstest (verschobene Antworten fallen auf ca. 48 %, siehe db_13).

In [ ]:
# B1: Vorhersagen laden (gesichert aus db_10) oder neu berechnen.
FENSTER = {"Entwicklung": DEV, "Test": TEST}
PROBA = {}
t0 = time.time()
for H in SERIES:
    for fenster, win in FENSTER.items():
        pfad = f"{OUT}/db10_proba_XGBoost_{H}_{fenster}.csv"
        if os.path.exists(pfad):
            PROBA[(H, fenster)] = pd.read_csv(pfad, parse_dates=["date"])
        else:
            print("Rechne neu:", SERIEN_NAMEN[H], fenster, f"({time.time() - t0:.0f}s)")
            res = walk_forward(d, H, build_features(d, H), win, entry_lag=0, feat_lag=0)
            res.to_csv(pfad, index=False)
            PROBA[(H, fenster)] = res
print("Vorhersagen bereit für", len(PROBA), "Läufe | Test: n =", sorted({len(v) for (h, f), v in PROBA.items() if f == 'Test'}))

## Teil C: Fehlermaße (Testzeitraum, Szenario S0)

In [ ]:
# C1: Alle Kennzahlen je Serie.
zeilen = []
for H in SERIES:
    F = build_features(d, H)
    r = evaluate_run(d, H, F, PROBA[(H, "Test")], 0, 0)
    r["Basisrate"] = PROBA[(H, "Test")]["y_true"].mean()
    zeilen.append(r)
R = pd.DataFrame(zeilen)
R["F1"] = 2 * R["Precision"] * R["Recall"] / (R["Precision"] + R["Recall"])
R["Spezifitaet"] = R["TN"] / (R["TN"] + R["FP"])            # richtig erkannte "nicht gestiegen"-Tage
R["Fehlalarmrate"] = R["FP"] / (R["FP"] + R["TN"])           # Kaufsignal, obwohl Preis nicht stieg
R["p_Holm"] = holm(R["p_nicht_ueberl"].values)
R["p_Holm_NYMEX"] = holm(R["p_McNemar_NYMEX"].values)
R["Serie_Name"] = R["Serie"].map(SERIEN_NAMEN)

print("=== 1. Trefferquote und wie sicher sie ist ===")
print(R[["Serie_Name", "n", "DA", "Wilson_lo", "Wilson_hi", "Boot_lo", "Boot_hi", "p_nicht_ueberl", "p_Holm"]]
      .rename(columns={"Serie_Name": "Serie", "DA": "Trefferquote"}).round(3).to_string(index=False))
print("\n=== 2. Vergleich mit einfachen Regeln ===")
print(R[["Serie_Name", "DA", "Mehrheit", "DA_NYMEX_Regel", "DA_Momentum", "p_Holm_NYMEX", "Brier_Skill"]]
      .rename(columns={"Serie_Name": "Serie", "DA": "Modell", "Mehrheit": "Immer häufigere Richtung", "DA_NYMEX_Regel": "NYMEX-Regel",
                       "DA_Momentum": "Momentum", "p_Holm_NYMEX": "p (Modell besser als NYMEX, Holm)"}).round(3).to_string(index=False))
print("\n=== 3. Fehlerarten ===")
print(R[["Serie_Name", "TP", "FP", "FN", "TN", "Precision", "Recall", "F1", "Spezifitaet", "Fehlalarmrate", "Basisrate"]]
      .rename(columns={"Serie_Name": "Serie", "TP": "Richtig gekauft", "FP": "Zu früh gekauft", "FN": "Zu lange gewartet",
                       "TN": "Richtig gewartet"}).round(3).to_string(index=False))
print(f"\nMittel über 8 Serien: Trefferquote {R['DA'].mean():.3f} | Precision {R['Precision'].mean():.3f} | Recall {R['Recall'].mean():.3f} | "
      f"Basisrate {R['Basisrate'].mean():.3f} | Brier-Skill {R['Brier_Skill'].mean():.3f}")
print("Lesehilfe: Precision = von den Kaufsignalen stieg der Preis wirklich. Recall = von den Anstiegen wurden so viele erkannt.")
print("Brier-Skill > 0: Wahrscheinlichkeiten besser als eine konstante Basisrate. p_nicht_ueberl: strenger Test gegen überlappende Labels.")

In [ ]:
# C2: Konfusionsmatrizen (Fehlerübersicht).
fig = plots.konfusion_raster(R[["Serie", "TP", "FP", "FN", "TN", "DA"]], "Fehlerübersicht aller 8 Serien (Konfusionsmatrizen)")
zeige(fig, "C_konfusion_alle.png")

# Eine große Matrix zum Zeigen
plots._stil()
z = R[R["Serie"] == "heizoel_sued"].iloc[0]
fig, ax = plt.subplots(figsize=(7, 4.6), dpi=110)
fig.subplots_adjust(left=0.2, right=0.95, top=0.72, bottom=0.08)
plots.konfusion_einzeln(ax, z["TP"], z["FP"], z["FN"], z["TN"], gross=True)
plots._kopf(fig, "Heizöl Süd: Wo das Modell richtig und falsch liegt",
            f"Test, n = {int(z['n'])} Tage. Trefferquote {z['DA']*100:.1f} %, Precision {z['Precision']*100:.0f} %, Recall {z['Recall']*100:.0f} %".replace(".", ","))
zeige(fig, "C_konfusion_heizoel_sued.png")

In [ ]:
# C3: Trefferquote im Vergleich zu einfachen Regeln.
fig = plots.trefferquote_mit_baselines(R[["Serie", "DA", "Wilson_lo", "Wilson_hi", "Mehrheit", "DA_NYMEX_Regel"]],
                                       "Trefferquote je Serie im Test")
zeige(fig, "C_trefferquote.png")

In [ ]:
# C4: Precision, Recall, Basisrate.
fig = plots.precision_recall(R[["Serie", "Precision", "Recall", "Basisrate"]], "Ein Kaufsignal trifft öfter als ein beliebiger Tag")
zeige(fig, "C_precision_recall.png")

In [ ]:
# C5: Kalibrierung. Kann man der Wahrscheinlichkeit des Modells trauen?
p_alle = np.concatenate([PROBA[(H, "Test")]["proba"].values for H in SERIES])
y_alle = np.concatenate([PROBA[(H, "Test")]["y_true"].values for H in SERIES])
fig = plots.kalibrierung(p_alle, y_alle, sub="Alle 8 Serien zusammen, Test. Die Serien sind verwandt, die Balken sind daher zu optimistisch.")
zeige(fig, "C_kalibrierung.png")

In [ ]:
# C6: Kosten gegenüber "immer kaufen".
print(R[["Serie_Name", "Regret_Modell", "Regret_immer_kaufen", "Ersparnis_vs_kaufen", "Ersp_lo", "Ersp_hi",
         "davon_zu_frueh_gekauft", "davon_zu_lange_gewartet"]]
      .rename(columns={"Serie_Name": "Serie", "Regret_Modell": "Mehrkosten Modell", "Regret_immer_kaufen": "Mehrkosten immer kaufen",
                       "Ersparnis_vs_kaufen": "Ersparnis", "davon_zu_frueh_gekauft": "davon zu früh gekauft",
                       "davon_zu_lange_gewartet": "davon zu lange gewartet"}).round(3).to_string(index=False))
print("\nEuro pro 100 l und Tag gegenüber dem besten Zeitpunkt (heute oder in 3 Tagen). Ersparnis > 0: Modell ist günstiger als immer kaufen.")
print(f"Serien mit abgesicherter Ersparnis (untere Grenze > 0): {int((R['Ersp_lo'] > 0).sum())} von {len(R)}")
fig = plots.ersparnis(R[["Serie", "Ersparnis_vs_kaufen", "Ersp_lo", "Ersp_hi"]])
zeige(fig, "C_ersparnis.png")

## Teil D: Dritte Stufe „Beobachten“

**Regel:** Kaufen, wenn p ≥ 0,5 + δ. Warten, wenn p < 0,5 − δ. **Beobachten** dazwischen (das Modell ist unsicher). δ wird nur aus dem **Entwicklungszeitraum** abgeleitet, danach einmalig im Test geprüft. Aus 20, 30 und 40 % Beobachten wählt die Regel den Anteil mit der besten Trefferquote auf der Entwicklung.

In [ ]:
# D1: Anteil Beobachten wählen und prüfen.
Q_GRID, Q_ERLAUBT = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5], [0.2, 0.3, 0.4]
p_dev = np.concatenate([PROBA[(H, "Entwicklung")]["proba"].values for H in SERIES])
absdev = np.abs(p_dev - 0.5)
zeilen = []
for q in Q_GRID:
    delta = float(np.quantile(absdev, q))
    for fenster in FENSTER:
        r = pd.DataFrame([bewerte_dreistufig(d, H, PROBA[(H, fenster)], delta) for H in SERIES])
        zeilen.append({"Anteil_Beobachten": q, "delta": delta, "Fenster": fenster, "Abdeckung": r["Abdeckung"].mean(),
                       "DA_entschieden": r["DA_entschieden"].mean(), "Precision_Kaufen": r["Precision_Kaufen"].mean(),
                       "Precision_Warten": r["Precision_Warten"].mean(), "Ersparnis_A": r["Ersparnis_A"].mean(),
                       "Ersparnis_B": r["Ersparnis_B"].mean()})
T = pd.DataFrame(zeilen)
for fenster in FENSTER:
    print(f"=== {fenster} (Mittel über 8 Serien) ===")
    print(T[T["Fenster"] == fenster].drop(columns="Fenster").round(3).to_string(index=False))
dev = T[(T["Fenster"] == "Entwicklung") & (T["Anteil_Beobachten"].isin(Q_ERLAUBT))]
q_star = float(dev.sort_values("DA_entschieden", ascending=False)["Anteil_Beobachten"].iloc[0])
delta_star = float(T[T["Anteil_Beobachten"] == q_star]["delta"].iloc[0])
test0 = T[(T["Anteil_Beobachten"] == 0.0) & (T["Fenster"] == "Test")].iloc[0]
testq = T[(T["Anteil_Beobachten"] == q_star) & (T["Fenster"] == "Test")].iloc[0]
print(f"\nAuf der Entwicklung gewählt: {q_star:.0%} Beobachten (δ = {delta_star:.3f})")
print(f"Test: Trefferquote {test0.DA_entschieden:.3f} -> {testq.DA_entschieden:.3f} auf {testq.Abdeckung:.0%} der Tage. "
      f"Ersparnis (A) {test0.Ersparnis_A:.3f} -> {testq.Ersparnis_A:.3f} Euro pro 100 l und Tag.")
fig = plots.abdeckung_kurve(T[["Fenster", "Abdeckung", "DA_entschieden"]])
zeige(fig, "D_beobachten_kurve.png")

In [ ]:
# D2: So sieht das tägliche Signal aus (letzte 10 Tage des Testzeitraums, Heizöl).
REGION = {"heizoel_sued": "Süd", "heizoel_suedwest": "Südwest", "heizoel_suedost": "Südost", "heizoel_rheinmain": "Rhein-Main"}
zl = []
for H, reg in REGION.items():
    res = PROBA[(H, "Test")].tail(10)
    for dt, p, s in zip(res["date"], res["proba"], signal(res["proba"].values, delta_star)):
        zl.append({"Datum": dt.date(), "Region": "Heizöl " + reg, "Signal": NAMEN[int(s)]})
print(pd.DataFrame(zl).pivot(index="Datum", columns="Region", values="Signal").to_string())
print("\nBeispiel aus dem Testzeitraum, kein aktuelles Signal. Die Regionen bewegen sich fast gleich: im Kern ein Marktsignal.")

## Teil E: Neue Tage, die das Modell noch nie gesehen hat

**So funktioniert es:** Das Modell wird mit allen bisherigen Daten trainiert und sagt dann die neuen Tage voraus, Tag für Tag, ohne deren Ergebnis zu kennen. Danach wird verglichen, was wirklich passiert ist.

**Was du eingeben kannst:** Eine Datei mit Argus-Preisen für neuere Tage (CSV mit den Spalten `date`, den 8 Preisreihen und `ice_gasoil`, oder eine Argus-Excel im selben Format wie die bisherige). Standardmäßig wird die neueste Datei `argus_api_*.csv` aus dem Output-Ordner genommen. Eigene Datei: weiter unten bei `NEU_ARGUS_DATEI` den Pfad eintragen. Die Marktdaten (Brent, WTI, Heizöl, Dollar-Euro) kommen aus der Marktdaten-Excel und müssen die neuen Tage abdecken.

**Hinweis zur Einordnung:** Dieser Bereich ist ein **erster Blick auf neue Tage**. Die formale Prüfung (Pre-Registration, Holdout A) ist noch nicht eingefroren. Wer das Ergebnis streng belegen will, setzt `NEU_AUSWERTEN = False`, dann werden nur Signale erzeugt und keine Trefferquote berechnet.

In [ ]:
# E1: Einstellungen und neue Daten laden.
NEU_ARGUS_DATEI = suche_neueste_api_csv(OUT) or ""     # eigenen Pfad eintragen, z. B. f"{BASE_PATH}/meine_neuen_daten.csv"
NEU_AUSWERTEN = True                                    # False = nur Signale, keine Bewertung
SERIEN_NEU = list(SERIES)                               # z. B. ["heizoel_sued"] für einen schnellen Lauf
DELTA_NEU = delta_star                                  # Breite des Beobachten-Bereichs aus Teil D

NEU_OK = False
if not NEU_ARGUS_DATEI or not os.path.exists(NEU_ARGUS_DATEI):
    print("Keine Datei mit neuen Daten gefunden.")
    print(f"Lege eine CSV oder Excel in {OUT} ab (CSV: Spalten date, die 8 Serien und ice_gasoil) und trage den Pfad oben bei NEU_ARGUS_DATEI ein.")
else:
    try:
        neu = lade_neue_argus(NEU_ARGUS_DATEI)
        argus_komb, ber = kombiniere_argus(argus, neu)
        d_neu_all, _ = build_dataset(argus_komb, markt)
        ende_markt = min(m["date"].max() for m in markt)          # letzter Tag, an dem ALLE Marktreihen echte Werte haben
        d_neu = d_neu_all[d_neu_all["date"] <= ende_markt].reset_index(drop=True)
        print(f"Marktdaten enden am {ende_markt.date()}. Spätere Tage werden nicht benutzt (sonst wären die Marktwerte veraltet).")
        START_NEU = LAST_ARGUS + pd.Timedelta(days=1)
        n_neu = int((d_neu["date"] >= START_NEU).sum())
        print("Datei:", os.path.basename(NEU_ARGUS_DATEI))
        print(f"Überlappung mit bisherigen Daten: {ber['Tage_ueberlappung']} Tage, größte Abweichung {ber['max_Abweichung_Ueberlappung']}")
        print(f"Neue Tage mit Argus-Daten: {ber['Tage_neu_dazu']} | davon mit vollständigen Marktdaten nutzbar: {n_neu}")
        print(f"Neue Tage im Modell: {d_neu.loc[d_neu['date'] >= START_NEU, 'date'].min().date() if n_neu else '-'} bis {d_neu['date'].max().date()}")
        if n_neu < 10:
            print("Zu wenige neue Tage mit Marktdaten. Prüfe, ob die Marktdaten-Excel die neuen Tage abdeckt.")
        else:
            for H in SERIES:
                leak_test(d_neu, H)
            print("Leakage-Test auf den erweiterten Daten bestanden.")
            NEU_OK = True
    except Exception as e:
        print("Die neuen Daten konnten nicht geladen werden:", e)

In [ ]:
# E2: Signale für die neuen Tage (ohne deren Ergebnis zu benutzen).
if NEU_OK:
    ENDE_NEU = d_neu["date"].max()
    ALLE_NEU, F_NEU, t0 = {}, {}, time.time()
    for H in SERIEN_NEU:
        res, F = signale_neue_tage(d_neu, H, START_NEU, ENDE_NEU, DELTA_NEU)
        ALLE_NEU[H], F_NEU[H] = res, F
        print(f"{SERIEN_NAMEN[H]} fertig ({time.time() - t0:.0f}s)")
    sig_alle = pd.concat(ALLE_NEU.values())
    sig_alle.to_csv(f"{OUT}/db15_signale_neue_tage.csv", index=False)
    print("\nVerteilung der Signale auf den neuen Tagen (alle Serien):")
    print(sig_alle["Signal"].value_counts().to_string())
    print("\nLetzte 12 Tage mit Signal je Serie (Ergebnis steht bei den letzten 3 Tagen noch aus):")
    letzte = sig_alle.groupby("Serie").tail(12).pivot(index="date", columns="Serie", values="Signal")
    letzte.columns = [SERIEN_NAMEN[c] for c in letzte.columns]
    print(letzte.to_string())
else:
    print("Übersprungen (keine nutzbaren neuen Daten, siehe oben).")

In [ ]:
# E3: Bewertung auf den neuen Tagen: wie viele Signale waren richtig?
if NEU_OK and NEU_AUSWERTEN:
    zeilen = []
    for H in SERIEN_NEU:
        res = ALLE_NEU[H]
        bekannt = res[res["Ergebnis_bekannt"]][["date", "y_true", "proba"]].reset_index(drop=True)
        r = evaluate_run(d_neu, H, F_NEU[H], bekannt, 0, 0)
        r["Basisrate"] = bekannt["y_true"].mean()
        dr = bewerte_dreistufig(d_neu, H, bekannt, DELTA_NEU)
        r["Abdeckung_3Stufen"], r["DA_sichere_Tage"] = dr["Abdeckung"], dr.get("DA_entschieden", np.nan)
        zeilen.append(r)
    RN = pd.DataFrame(zeilen)
    RN["Serie_Name"] = RN["Serie"].map(SERIEN_NAMEN)
    n_neu_bew = int(RN["n"].iloc[0])
    print("ERSTER BLICK AUF NEUE TAGE (explorativ, nicht die formale Prüfung)")
    print(f"Bewertet: {n_neu_bew} Tage je Serie ({bekannt['date'].min().date()} bis {bekannt['date'].max().date()}). Bei so wenigen Tagen ist die Unsicherheit groß.\n")
    print(RN[["Serie_Name", "n", "DA", "Wilson_lo", "Wilson_hi", "Boot_lo", "Boot_hi", "Mehrheit", "DA_NYMEX_Regel", "Precision", "Recall",
              "Basisrate", "Brier_Skill", "Ersparnis_vs_kaufen", "Ersp_lo"]]
          .rename(columns={"Serie_Name": "Serie", "DA": "Trefferquote"}).round(3).to_string(index=False))
    vergleich = pd.DataFrame({"Test (alt)": [R["DA"].mean(), R["Precision"].mean(), R["Recall"].mean()],
                              "Neue Tage": [RN["DA"].mean(), RN["Precision"].mean(), RN["Recall"].mean()]},
                             index=["Trefferquote", "Precision", "Recall"]).round(3)
    print("\nVergleich (Mittel über die Serien):")
    print(vergleich.to_string())
    print(f"\nMit Beobachten-Stufe (δ = {DELTA_NEU:.3f}): Trefferquote auf den sicheren Tagen {RN['DA_sichere_Tage'].mean():.3f} "
          f"bei {RN['Abdeckung_3Stufen'].mean():.0%} der Tage.")
    fig = plots.konfusion_raster(RN[["Serie", "TP", "FP", "FN", "TN", "DA"]], "Neue Tage: Fehlerübersicht (Konfusionsmatrizen)",
                                 f"Tage, die das Modell nie gesehen hat. n = {n_neu_bew} je Serie.")
    zeige(fig, "E_konfusion_neue_tage.png")
    fig = plots.trefferquote_mit_baselines(RN[["Serie", "DA", "Wilson_lo", "Wilson_hi", "Mehrheit", "DA_NYMEX_Regel"]],
                                           "Neue Tage: Trefferquote je Serie", f"n = {n_neu_bew} Tage je Serie, erster Blick (explorativ). Balken = 95-%-Intervall (Wilson).")
    zeige(fig, "E_trefferquote_neue_tage.png")
    print("\nEinordnung: Bei", n_neu_bew, "Tagen kann ein Test nur große Unterschiede zeigen. Ein Ergebnis unter dem alten Test heißt nicht automatisch, dass das Modell nicht funktioniert.")
    print("Liegt die Trefferquote auch hier klar über der Basisrate und den einfachen Regeln, ist das ein gutes Zeichen. Ein Beleg im strengen Sinn ist es erst nach der Pre-Registration.")
elif NEU_OK:
    print("Bewertung ist ausgeschaltet (NEU_AUSWERTEN = False). Es wurden nur Signale erzeugt.")
else:
    print("Übersprungen.")

## Teil F: Was das Ergebnis nicht zeigt

- **Zeitpunkt des Einkaufs:** Alles gilt für Szenario S0 (Kauf zum Preis von Tag t). Kann der Einkäufer erst am Folgetag kaufen, liegt die Trefferquote bei ca. 50 %. Offen ist auch, woher am Entscheidungstag die US-Preise kommen (die Marktdaten sehen nach verzögert veröffentlichten Spotpreisen aus).
- **Statistik:** Die Serien sind stark verwandt und zählen nicht als 8 unabhängige Belege. Die 3-Tage-Ergebnisse überlappen, deshalb sind einfache Konfidenzintervalle zu optimistisch.
- **Vergleich mit Regeln:** Die einfache NYMEX-Regel erreicht schon 57–61 %. Der Vorsprung des Modells ist klein und nach Holm-Korrektur nicht signifikant.
- **Beobachten:** Macht die Signale zuverlässiger, spart aber im Durchschnitt nicht mehr Geld.
- **Neue Tage (Teil E):** Wenige Tage, daher große Unsicherheit. Keine Ursachen-Aussage, nur Vorhersage-Güte.
- **Nicht enthalten:** Nachrichten, Rheinfracht, andere Horizonte (1 und 14 Tage).

In [ ]:
# Zusammenfassung zum Mitnehmen.
print("ZUSAMMENFASSUNG")
print(f"- Daten: {d.shape[0]} Tage bis {d['date'].max().date()}, 8 Preisreihen, {len(merkmale)} Eingaben je Serie.")
print(f"- Test (n = {int(R['n'].iloc[0])} Tage je Serie): Trefferquote {R['DA'].mean():.1%} im Mittel ({R['DA'].min():.1%} bis {R['DA'].max():.1%}); einfache NYMEX-Regel {R['DA_NYMEX_Regel'].mean():.1%}.")
print(f"- Precision {R['Precision'].mean():.1%} (Basisrate {R['Basisrate'].mean():.1%}), Recall {R['Recall'].mean():.1%}.")
print(f"- Ersparnis gegenüber 'immer kaufen': {R['Ersparnis_vs_kaufen'].mean():.2f} Euro pro 100 l und Tag, bei {int((R['Ersp_lo'] > 0).sum())} von 8 Serien abgesichert.")
print(f"- Mit Beobachten ({q_star:.0%} der Tage): Trefferquote auf den übrigen Tagen {testq.DA_entschieden:.1%}.")
if NEU_OK and NEU_AUSWERTEN:
    print(f"- Neue Tage (explorativ, {n_neu_bew} Tage): Trefferquote {RN['DA'].mean():.1%} im Mittel.")
print(f"- Grafiken gespeichert in: {GRAFIK}")